# W2 Physical Identity Probe

这个 notebook **不接入上位机**，只用于实验前验证 RunE W2 的物理设备身份来源。

目标：

1. 记录 Windows / USB 串口层能够看到的 `VID / PID / serial_number / location / hwid`；
2. 通过当前 W2 协议读取 `software (0x02)`、`device name (0x03)`、`hardware version (0x1D)`；
3. 保留每条命令和设备原始响应 hex，不预设未知的设备信息 payload 格式；
4. 比较多台 W2、换 USB 口、换电脑之后哪些字段稳定，从而决定最终使用 `protocol identity / USB serial / manual registry`。

> 安全边界：本 notebook 不尝试修改 device name。查询前仅发送项目中已经使用的 `stop_collect()`，避免连续采集数据淹没查询响应。请先关闭正在占用 W2 串口的上位机/其他程序。


In [1]:
from __future__ import annotations

import hashlib
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import serial
from serial.tools import list_ports
from IPython.display import display

# 允许从 repo root 或 assembly/testers 目录启动 Jupyter。
cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / 'DeviceInterface' / 'w2_protocol.py').exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repository root containing DeviceInterface/w2_protocol.py')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from DeviceInterface.w2_protocol import (
    W2CommandBuilder,
    W2_NOTIFY_HEADER,
    W2_NOTIFY_TAIL,
)

print('REPO_ROOT:', REPO_ROOT)
print('pyserial :', serial.VERSION)


REPO_ROOT: E:\ALL4gdt\EMGacq\emgchachito
pyserial : 3.5


## 1. 枚举当前串口 / USB 硬件信息

重点观察：

- `serial_number`：如果每块 W2 都不同且换 USB 口后保持不变，它可以成为很强的 hardware identity 候选；
- `vid/pid`：通常只能说明设备型号，不能区分同型号的多块 W2；
- `location`：通常描述 USB 拓扑位置，换接口/Hub/电脑后可能变化，只适合辅助定位；
- `COMx`：只作为当前 transport locator，不应作为设备身份。


In [2]:
def port_info_dict(port) -> dict[str, object]:
    return {
        'device': port.device,
        'description': port.description,
        'hwid': port.hwid,
        'vid': port.vid,
        'pid': port.pid,
        'serial_number': port.serial_number,
        'location': port.location,
        'manufacturer': port.manufacturer,
        'product': port.product,
        'interface': port.interface,
    }

PORT_INVENTORY = [port_info_dict(port) for port in list_ports.comports()]
ports_df = pd.DataFrame(PORT_INVENTORY)
display(ports_df if not ports_df.empty else pd.DataFrame(columns=[
    'device', 'description', 'hwid', 'vid', 'pid', 'serial_number',
    'location', 'manufacturer', 'product', 'interface'
]))

if not PORT_INVENTORY:
    print('No serial ports detected.')


,device,description,hwid,vid,pid,serial_number,location,manufacturer,product,interface
0,COM13,USB-SERIAL CH340 (COM13),USB VID:PID=1A86:7523 SER= LOCATION=1-4.2.1,6790,29987,,1-4.2.1,wch.cn,None,None
1,COM10,USB-SERIAL CH340 (COM10),USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.1,6790,29987,,1-4.1.1,wch.cn,None,None
2,COM11,USB-SERIAL CH340 (COM11),USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.2,6790,29987,,1-4.1.2,wch.cn,None,None
3,COM12,USB-SERIAL CH340 (COM12),USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.3,6790,29987,,1-4.1.3,wch.cn,None,None
4,COM9,USB-SERIAL CH340 (COM9),USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.4,6790,29987,,1-4.1.4,wch.cn,None,None


## 2. 指定本轮要验证的 W2

先根据上一个 cell 的输出填写 `PORTS_TO_PROBE`。建议一次把当前接入的 W2 全部列出。

例如：

```python
PORTS_TO_PROBE = ['COM9', 'COM11', 'COM13']
```

不要让上位机同时占用这些串口。


In [3]:
PORTS_TO_PROBE: list[str] = ['COM9', 'COM10','COM11', 'COM12','COM13']  # TODO: e.g. ['COM9', 'COM11']

BAUD_RATE = 256000
SERIAL_TIMEOUT_S = 0.05
STOP_SETTLE_S = 0.15
RESPONSE_WINDOW_S = 1.0
IDLE_AFTER_FIRST_BYTE_S = 0.15

QUERY_ADDRESSES = {
    'software': W2CommandBuilder.ADDRESS_SOFTWARE,
    'device_name': W2CommandBuilder.ADDRESS_DEVICE_NAME,
    'hardware_version': W2CommandBuilder.ADDRESS_HW_VERSION,
}

print('Ports to probe:', PORTS_TO_PROBE or '<fill PORTS_TO_PROBE first>')
print('Read commands:')
for name, address in QUERY_ADDRESSES.items():
    command_hex = W2CommandBuilder.read(address).hex(' ').upper()
    print(f'  {name:16} address=0x{address:02X} command={command_hex}')


Ports to probe: ['COM9', 'COM10', 'COM11', 'COM12', 'COM13']
Read commands:
  software         address=0x02 command=AA 03 81 02 3B BB
  device_name      address=0x03 command=AA 03 81 03 3A BB
  hardware_version address=0x1D command=AA 03 81 1D 24 BB


## 3. 通用响应分帧与可读性检查

当前项目里的 `W2StreamParser` 只正式解释 `0x11` 数据采集帧，因此这里**不强行解释设备信息 payload**。

这里只依据现有 W2 notify envelope 做通用分帧：`A5 / length / frame_type / checksum / payload / 5A`。每个 frame 都会保留：

- `frame_type`；
- checksum / tail 是否符合当前协议；
- payload hex；
- 去掉 `\x00` 后的 ASCII / UTF-8 尝试。

如果设备返回格式不同，`raw_hex` 仍然会完整保留，后续可以据此扩展 parser。


In [4]:
def _printable_ascii(data: bytes) -> str:
    trimmed = data.strip(b'\x00')
    return ''.join(chr(b) if 32 <= b <= 126 else '.' for b in trimmed)


def _utf8_attempt(data: bytes) -> str | None:
    trimmed = data.strip(b'\x00')
    if not trimmed:
        return ''
    try:
        return trimmed.decode('utf-8')
    except UnicodeDecodeError:
        return None


def split_w2_notify_frames(raw: bytes) -> tuple[list[dict[str, object]], bytes]:
    frames: list[dict[str, object]] = []
    buffer = bytearray(raw)
    skipped = bytearray()

    while buffer:
        try:
            header_index = buffer.index(W2_NOTIFY_HEADER)
        except ValueError:
            skipped.extend(buffer)
            buffer.clear()
            break

        if header_index:
            skipped.extend(buffer[:header_index])
            del buffer[:header_index]

        if len(buffer) < 2:
            skipped.extend(buffer)
            break

        frame_len = int(buffer[1]) + 3
        if frame_len < 6 or len(buffer) < frame_len:
            skipped.extend(buffer)
            break

        frame = bytes(buffer[:frame_len])
        del buffer[:frame_len]
        payload = frame[4:-1]
        frames.append({
            'frame_hex': frame.hex(' ').upper(),
            'length_field': frame[1],
            'frame_type': frame[2],
            'frame_type_hex': f'0x{frame[2]:02X}',
            'checksum_ok': frame[3] == (frame[1] ^ frame[2]),
            'tail_ok': frame[-1] == W2_NOTIFY_TAIL,
            'payload_hex': payload.hex(' ').upper(),
            'payload_ascii': _printable_ascii(payload),
            'payload_utf8': _utf8_attempt(payload),
        })

    return frames, bytes(skipped)


In [5]:
def _read_response_bytes(
    handle: serial.Serial,
    *,
    total_window_s: float = RESPONSE_WINDOW_S,
    idle_after_first_byte_s: float = IDLE_AFTER_FIRST_BYTE_S,
) -> bytes:
    deadline = time.monotonic() + total_window_s
    last_data_at: float | None = None
    chunks: list[bytes] = []

    while time.monotonic() < deadline:
        waiting = int(getattr(handle, 'in_waiting', 0) or 0)
        chunk = bytes(handle.read(waiting if waiting > 0 else 1))
        if chunk:
            chunks.append(chunk)
            last_data_at = time.monotonic()
            continue
        if last_data_at is not None and time.monotonic() - last_data_at >= idle_after_first_byte_s:
            break

    return b''.join(chunks)


def _drain_for(handle: serial.Serial, duration_s: float) -> bytes:
    deadline = time.monotonic() + duration_s
    chunks: list[bytes] = []
    while time.monotonic() < deadline:
        waiting = int(getattr(handle, 'in_waiting', 0) or 0)
        chunk = bytes(handle.read(waiting if waiting > 0 else 1))
        if chunk:
            chunks.append(chunk)
    return b''.join(chunks)


def probe_w2_port(port: str) -> dict[str, object]:
    result: dict[str, object] = {
        'port': port,
        'queries': {},
        'error': None,
    }

    try:
        with serial.Serial(
            port,
            BAUD_RATE,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            timeout=SERIAL_TIMEOUT_S,
        ) as handle:
            handle.reset_input_buffer()
            handle.reset_output_buffer()

            # 已知协议命令：只停止连续采集，不改变持久配置。
            stop_command = W2CommandBuilder.stop_collect()
            handle.write(stop_command)
            handle.flush()
            stop_response = _drain_for(handle, STOP_SETTLE_S)
            result['stop_collect'] = {
                'command_hex': stop_command.hex(' ').upper(),
                'response_hex': stop_response.hex(' ').upper(),
            }
            handle.reset_input_buffer()

            queries: dict[str, object] = {}
            for name, address in QUERY_ADDRESSES.items():
                handle.reset_input_buffer()
                command = W2CommandBuilder.read(address)
                handle.write(command)
                handle.flush()
                raw = _read_response_bytes(handle)
                frames, skipped = split_w2_notify_frames(raw)
                queries[name] = {
                    'address': address,
                    'address_hex': f'0x{address:02X}',
                    'command_hex': command.hex(' ').upper(),
                    'raw_hex': raw.hex(' ').upper(),
                    'raw_len': len(raw),
                    'response_sha256': hashlib.sha256(raw).hexdigest() if raw else None,
                    'frames': frames,
                    'unparsed_hex': skipped.hex(' ').upper(),
                }
                time.sleep(0.05)

            result['queries'] = queries
    except Exception as exc:
        result['error'] = f'{type(exc).__name__}: {exc}'

    return result


## 4. 执行查询并检查原始返回

第一次建议只测一块 W2，确认没有串口占用和异常；之后把所有 W2 一起列入 `PORTS_TO_PROBE` 比较。


In [6]:
PROBE_RESULTS: dict[str, dict[str, object]] = {}

if not PORTS_TO_PROBE:
    print('PORTS_TO_PROBE is empty. Fill it in cell 2, then rerun from there.')
else:
    for port in PORTS_TO_PROBE:
        print(f'\n=== Probing {port} ===')
        result = probe_w2_port(port)
        PROBE_RESULTS[port] = result
        if result['error']:
            print('ERROR:', result['error'])
            continue
        for query_name, query in result['queries'].items():
            print(f'\n[{query_name}] {query["address_hex"]}')
            print('command :', query['command_hex'])
            print('raw     :', query['raw_hex'] or '<no bytes>')
            print('frames  :', len(query['frames']))
            for index, frame in enumerate(query['frames']):
                print(
                    f'  frame[{index}] type={frame["frame_type_hex"]} '
                    f'checksum_ok={frame["checksum_ok"]} tail_ok={frame["tail_ok"]}'
                )
                print('    payload_hex  :', frame['payload_hex'])
                print('    payload_ascii:', repr(frame['payload_ascii']))
                print('    payload_utf8 :', repr(frame['payload_utf8']))
            if query['unparsed_hex']:
                print('unparsed:', query['unparsed_hex'])



=== Probing COM9 ===

[software] 0x02
command : AA 03 81 02 3B BB
raw     : A5 0A 02 08 57 32 5F 56 31 2E 36 00 5A
frames  : 1
  frame[0] type=0x02 checksum_ok=True tail_ok=True
    payload_hex  : 57 32 5F 56 31 2E 36 00
    payload_ascii: 'W2_V1.6'
    payload_utf8 : 'W2_V1.6'

[device_name] 0x03
command : AA 03 81 03 3A BB
raw     : A5 0B 03 08 52 75 6E 45 20 57 32 20 32 5A
frames  : 1
  frame[0] type=0x03 checksum_ok=True tail_ok=True
    payload_hex  : 52 75 6E 45 20 57 32 20 32
    payload_ascii: 'RunE W2 2'
    payload_utf8 : 'RunE W2 2'

[hardware_version] 0x1D
command : AA 03 81 1D 24 BB
raw     : A5 03 1D 1E 03 5A
frames  : 1
  frame[0] type=0x1D checksum_ok=True tail_ok=True
    payload_hex  : 03
    payload_ascii: '.'
    payload_utf8 : '\x03'

=== Probing COM10 ===

[software] 0x02
command : AA 03 81 02 3B BB
raw     : A5 0A 02 08 57 32 5F 56 31 2E 36 00 5A
frames  : 1
  frame[0] type=0x02 checksum_ok=True tail_ok=True
    payload_hex  : 57 32 5F 56 31 2E 36 00
    payload

## 5. 汇总比较不同 W2 的响应

这里不声称 payload 已经被正确语义解析，而是把同一查询的 response signature 并排放出来。

重点看：

- 不同物理 W2 的 `device_name` raw/payload 是否不同；
- 同一台 W2 换 USB 口之后 payload / response hash 是否保持；
- `software` / `hardware_version` 如果所有设备相同，只能作为型号/版本信息，不能作为 identity。


In [7]:
summary_rows: list[dict[str, object]] = []
inventory_by_port = {row['device']: row for row in PORT_INVENTORY}

for port, result in PROBE_RESULTS.items():
    usb = inventory_by_port.get(port, {})
    base = {
        'port': port,
        'usb_vid': usb.get('vid'),
        'usb_pid': usb.get('pid'),
        'usb_serial': usb.get('serial_number'),
        'usb_location': usb.get('location'),
        'usb_hwid': usb.get('hwid'),
        'error': result.get('error'),
    }
    for query_name, query in result.get('queries', {}).items():
        first_frame = query['frames'][0] if query['frames'] else {}
        summary_rows.append({
            **base,
            'query': query_name,
            'address': query['address_hex'],
            'raw_hex': query['raw_hex'],
            'response_sha256': query['response_sha256'],
            'first_frame_type': first_frame.get('frame_type_hex'),
            'first_payload_hex': first_frame.get('payload_hex'),
            'first_payload_ascii': first_frame.get('payload_ascii'),
            'first_payload_utf8': first_frame.get('payload_utf8'),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,port,usb_vid,usb_pid,usb_serial,usb_location,usb_hwid,error,query,address,raw_hex,response_sha256,first_frame_type,first_payload_hex,first_payload_ascii,first_payload_utf8
0,COM9,6790,29987,,1-4.1.4,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.4,None,software,0x02,A5 0A 02 08 57 32 5F 56 31 2E 36 00 5A,380e702aafab362810b4cca042a891e84d547a90e310ea...,0x02,57 32 5F 56 31 2E 36 00,W2_V1.6,W2_V1.6
1,COM9,6790,29987,,1-4.1.4,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.4,None,device_name,0x03,A5 0B 03 08 52 75 6E 45 20 57 32 20 32 5A,776ca1edca1ed87e50112f6fdb10333ecc3495e6405d3e...,0x03,52 75 6E 45 20 57 32 20 32,RunE W2 2,RunE W2 2
2,COM9,6790,29987,,1-4.1.4,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.4,None,hardware_version,0x1D,A5 03 1D 1E 03 5A,5f5a80074ae43b6228c6d4390ac895d6cc5f9fa21761d0...,0x1D,03,.,
3,COM10,6790,29987,,1-4.1.1,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.1,None,software,0x02,A5 0A 02 08 57 32 5F 56 31 2E 36 00 5A,380e702aafab362810b4cca042a891e84d547a90e310ea...,0x02,57 32 5F 56 31 2E 36 00,W2_V1.6,W2_V1.6
4,COM10,6790,29987,,1-4.1.1,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.1,None,device_name,0x03,A5 0B 03 08 52 75 6E 45 20 57 32 20 35 5A,c08f8e2f3debf30c19534ecdb9e788b368e5e255c55574...,0x03,52 75 6E 45 20 57 32 20 35,RunE W2 5,RunE W2 5
5,COM10,6790,29987,,1-4.1.1,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.1,None,hardware_version,0x1D,A5 03 1D 1E 03 5A,5f5a80074ae43b6228c6d4390ac895d6cc5f9fa21761d0...,0x1D,03,.,
6,COM11,6790,29987,,1-4.1.2,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.2,None,software,0x02,A5 0A 02 08 57 32 5F 56 31 2E 36 00 5A,380e702aafab362810b4cca042a891e84d547a90e310ea...,0x02,57 32 5F 56 31 2E 36 00,W2_V1.6,W2_V1.6
7,COM11,6790,29987,,1-4.1.2,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.2,None,device_name,0x03,A5 0B 03 08 52 75 6E 45 20 57 32 20 34 5A,ccaf3b218f32f53ed5f36646a26c522fc6c2659b28f080...,0x03,52 75 6E 45 20 57 32 20 34,RunE W2 4,RunE W2 4
8,COM11,6790,29987,,1-4.1.2,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.2,None,hardware_version,0x1D,A5 03 1D 1E 03 5A,5f5a80074ae43b6228c6d4390ac895d6cc5f9fa21761d0...,0x1D,03,.,
9,COM12,6790,29987,,1-4.1.3,USB VID:PID=1A86:7523 SER= LOCATION=1-4.1.3,None,software,0x02,A5 0A 02 08 57 32 5F 56 31 2E 36 00 5A,380e702aafab362810b4cca042a891e84d547a90e310ea...,0x02,57 32 5F 56 31 2E 36 00,W2_V1.6,W2_V1.6


## 6. 标记真实物理编号与本次测试环境

这一格只是给测试结果增加人工语义，不会写入 W2。

建议直接使用设备外壳上你能看到的真实编号，例如 `W2-01`、`W2-02`。如果当前设备没有标签，先临时贴纸编号。


In [8]:
PHYSICAL_LABELS: dict[str, str] = {
    # 'COM9': 'W2-01',
    # 'COM11': 'W2-02',
}

TEST_CONTEXT = {
    'machine': '',        # e.g. 'lab-pc-1'
    'usb_setup': '',      # e.g. 'front USB / hub port 2'
    'run_note': '',       # e.g. 'baseline before swapping USB ports'
}

print('Physical labels:', PHYSICAL_LABELS)
print('Test context   :', TEST_CONTEXT)


Physical labels: {}
Test context   : {'machine': '', 'usb_setup': '', 'run_note': ''}


## 7. 导出本轮完整证据

Notebook 本身会保存 cell output；这一格另外导出 JSON，便于后续把多次测试结果做 diff。

推荐至少保存三轮：

1. **baseline**：记录所有 W2；
2. **swap USB ports**：交换 USB 插口，再运行一次；
3. **another machine**（如果方便）：同一块 W2 换一台电脑再运行。

这样可以直接回答 USB serial、USB location、protocol device name 哪些是真正跨环境稳定的。


In [9]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = REPO_ROOT / 'captures' / 'w2_identity_probe'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'w2_identity_probe_{timestamp}.json'

report = {
    'created_local': datetime.now().astimezone().isoformat(),
    'test_context': TEST_CONTEXT,
    'physical_labels': PHYSICAL_LABELS,
    'serial_inventory': PORT_INVENTORY,
    'probe_ports': PORTS_TO_PROBE,
    'settings': {
        'baud_rate': BAUD_RATE,
        'serial_timeout_s': SERIAL_TIMEOUT_S,
        'response_window_s': RESPONSE_WINDOW_S,
        'idle_after_first_byte_s': IDLE_AFTER_FIRST_BYTE_S,
    },
    'results': PROBE_RESULTS,
}

output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', output_path)


Saved: E:\ALL4gdt\EMGacq\emgchachito\captures\w2_identity_probe\w2_identity_probe_20260903_144214.json


## 8. 如何据结果决定最终 identity 方案

完成多块设备和换口测试后，按下面的优先级判断：

1. **协议层存在每台设备唯一且稳定的值**（例如 `device_name` 实际就是唯一编号）  
   → 优先用 protocol identity，USB/COM 只负责找到连接。
2. **协议层不能区分设备，但 USB `serial_number` 每块唯一且换口/换电脑稳定**  
   → 用 USB serial 建立 `hardware identity -> logical W2 label` registry。
3. **USB serial 缺失/相同，协议也没有唯一值**  
   → 做一次人工物理标签绑定；`location` 可以辅助，但不能当设备本体 identity。
4. **绝不使用 COM 编号作为稳定设备 identity**。

最终实验语义仍然分三层：

```text
hardware identity -> logical device label -> experiment placement
                                    e.g. W2-01 -> FDI
```

等你实际运行并保存 notebook/JSON 后，可以把结果交给我继续判断 W2 的最终绑定策略。
